In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split
from scipy.stats import uniform, randint
import os
import joblib

# Load the preprocessed parquet data
# Ensure the 'date' column is in the correct format
df = pd.read_parquet('stocks.parquet')

# Convert the 'date' column to datetime objects
df['date'] = pd.to_datetime(df['date'])

# Set the years you want to train on
years_to_train = range(2015, 2025)

for year in years_to_train:
    print(f"\n--- Training model for the year: {year} ---")

    # Filter data for the specific year
    df_year = df[df['year'] == year]

    if df_year.empty:
        print(f"No data available for {year}. Skipping...")
        continue

    # Features and target
    X = df_year.drop(['id', 'date', 'ret_eom', 'gvkey', 'iid', 'excntry',
    'year', 'month', 'char_date', 'char_eom', 'stock_ret'], axis=1, errors='ignore')
    y = df_year['stock_ret']

    # Define the XGBoost regressor model
    xgbr = xgb.XGBRegressor(objective='reg:squarederror',
                            n_estimators=1500,
                            eval_metric='rmse',
                            early_stopping_rounds=50,
                            n_jobs=-1,
                            tree_method='gpu_hist' # This enables GPU acceleration
                           )

    # Define the parameter space for RandomizedSearchCV
    param_distributions = {
        'learning_rate': uniform(0.005, 0.3),
        'max_depth': randint(10, 15),
        'subsample': uniform(0.5, 0.4),
        'colsample_bytree': uniform(0.5, 0.4),
        'gamma': uniform(0, 0.5),
        'min_child_weight': randint(1, 10),
    }

    # Set up RandomizedSearchCV
    random_search = RandomizedSearchCV(
        xgbr,
        param_distributions=param_distributions,
        n_iter=25,
        cv=5,
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    # Split the data into training and testing sets for this year
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Perform the randomized search
    random_search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    # Get the best model from the search
    best_model = random_search.best_estimator_

    # Print the best parameters and score for the current year
    print(f"Best parameters for {year}: {random_search.best_params_}")
    print(f"Best cross-validation score for {year}: {-random_search.best_score_}")

    # Evaluate the best model on the test set for this year
    predictions = best_model.predict(X_test)
    test_mse = mean_squared_error(y_test, predictions)
    print(f"Model Mean Squared Error (MSE) on the test set for {year}: {test_mse}")

    # Create a DataFrame to save predictions and actual values
    results_df = pd.DataFrame({
        'actual_values': y_test,
        'predicted_values': predictions
    })

    # Save the results to a CSV file
    results_filename = f'predictions_{year}.csv'
    results_df.to_csv(results_filename, index=False)
    print(f"Predictions for {year} saved as {results_filename}")

    # Save the best model for this year
    model_filename = f'xgboost_model_{year}.joblib'
    joblib.dump(best_model, model_filename)
    print(f"Model for {year} saved as {model_filename}")

print("\nAll models have been trained and saved.")

In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split
from scipy.stats import uniform, randint
import os
import joblib

def train_and_save_model_for_year(filepath, year_to_train):
    """
    Loads data, then tunes, trains, and saves an XGBoost model for a given year.

    Args:
        filepath (str): The path to the parquet file.
        year_to_train (int): The specific year to train the model on.

    Returns:
        The trained XGBoost model object, or None if the file is not found or data is empty.
    """
    print(f"\n--- Training model for the year: {year_to_train} ---")
    
    # Load data from the specified file inside the function
    if not os.path.exists(filepath):
        print(f"Error: File not found at {filepath}")
        return None
        
    main_df = pd.read_parquet(filepath)
    main_df['date'] = pd.to_datetime(main_df['date'])

    # Filter data for the specific year
    df_year = main_df[main_df['date'].dt.year == year_to_train]

    if df_year.empty:
        print(f"No data available for {year_to_train}. Skipping...")
        return None

    # Define Features (X) and Target (y)
    X = df_year.drop(['id', 'date', 'ret_eom', 'gvkey', 'iid', 'excntry',
                      'year', 'month', 'char_date', 'char_eom', 'stock_ret'], axis=1, errors='ignore')
    y = df_year['stock_ret']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Define the XGBoost regressor model
    xgbr = xgb.XGBRegressor(objective='reg:squarederror',
                            n_estimators=1500,
                            eval_metric='rmse',
                            early_stopping_rounds=50,
                            n_jobs=-1,
                            tree_method='gpu_hist' # This enables GPU acceleration
                           )

    # Define the parameter space for RandomizedSearchCV
    param_distributions = {
        'learning_rate': uniform(0.005, 0.3),
        'max_depth': randint(10, 15),
        'subsample': uniform(0.5, 0.4),
        'colsample_bytree': uniform(0.5, 0.4),
        'gamma': uniform(0, 0.5),
        'min_child_weight': randint(1, 10),
    }

    # Set up RandomizedSearchCV
    random_search = RandomizedSearchCV(
        xgbr,
        param_distributions=param_distributions,
        n_iter=25,
        cv=5,
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    random_search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    # Get the best model and evaluate it
    best_model = random_search.best_estimator_
    predictions = best_model.predict(X_test)
    test_mse = mean_squared_error(y_test, predictions)

    print(f"Best parameters for {year_to_train}: {random_search.best_params_}")
    print(f"Model Mean Squared Error (MSE) on the test set for {year_to_train}: {test_mse:.6f}")

    # Save the model to a file
    model_filename = f'xgboost_model_{year_to_train}.joblib'
    joblib.dump(best_model, model_filename)
    print(f"Model for {year_to_train} saved as {model_filename}")

    # Return the trained model object
    return best_model